# 1단계: MVTec AD 재현

PatchCore(원본 코드)와 PaDiM(직접 구현)이 원 논문 값과 맞는지 확인한다. 설정 4개 x 15 부류 x 시드 3개, 모두 180번 돈다.

| 설정 | 이미지 AUROC | 픽셀 AUROC | PRO |
|---|---:|---:|---:|
| PatchCore-10%, WRN-50-2 (2,3) | 99.0 | 98.1 | 93.5 |
| PatchCore-1%, WRN-50-2 (2,3) | 99.0 | 98.0 | 93.1 |
| PaDiM, WRN-50-2 (차원 550) | 95.3 | 97.5 | 92.1 |
| PaDiM, ResNet18 (차원 100) | 논문에 없음 | 96.7 | 90.1 |

판정 폭은 이미지 AUROC ±0.5, 픽셀 AUROC와 PRO ±1.0이다. 맞지 않으면 2단계로 가지 않는다.

**실행 전 준비 둘.**
1. 런타임 유형을 GPU(L4)로 바꾼다.
2. 레포가 비공개이므로, 왼쪽 열쇠 아이콘(보안 비밀)에 `GITHUB_TOKEN`을 넣고 이 노트북에 접근을 허용한다. 레포 읽기 권한만 있는 토큰이면 된다.

끊겨도 된다. 결과는 부류 하나가 끝날 때마다 Drive의 CSV에 적고, 같은 셀을 다시 돌리면 끝난 줄을 건너뛴다.

## 1. 코드와 패키지

In [7]:
import os, subprocess, sys

REPO_DIR = "/content/beyond-mvtec"
UPSTREAM_DIR = "/content/patchcore-inspection"
UPSTREAM_COMMIT = "fcaa92f124fb1ad74a7acf56726decd4b27cbcad"

if not os.path.isdir(REPO_DIR):
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/chlee1027/beyond-mvtec.git", REPO_DIR], check=True)
    # 토큰이 설정 파일에 남지 않게 지운다
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", "https://github.com/chlee1027/beyond-mvtec.git"], check=True)
if not os.path.isdir(UPSTREAM_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/amazon-science/patchcore-inspection.git", UPSTREAM_DIR], check=True)
subprocess.run(["git", "-C", UPSTREAM_DIR, "checkout", "-q", UPSTREAM_COMMIT], check=True)

!pip uninstall -y -q sympy
!pip install -q sympy==1.13.3 faiss-gpu-cu12 timm click pretrainedmodels scikit-image scikit-learn scipy tqdm pandas pyyaml

sys.path[:0] = [os.path.join(REPO_DIR, "src"), os.path.join(UPSTREAM_DIR, "src")]
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

import faiss, torch, torchvision
assert torch.cuda.is_available(), "GPU 런타임이 아니다"
print(f"레포 {COMMIT} | torch {torch.__version__} | torchvision {torchvision.__version__} | faiss {faiss.__version__}")
print(f"GPU {torch.cuda.get_device_name(0)} | VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert hasattr(faiss, "StandardGpuResources"), "faiss 의 GPU 판이 아니다"

레포 981e1f6 | torch 2.14.1+cu130 | torchvision 0.29.1+cu130 | faiss 1.14.1
GPU NVIDIA L4 | VRAM 23.7 GB


## 2. Drive와 데이터

In [8]:
import shutil
from google.colab import drive
drive.mount("/content/drive")

MY = "/content/drive/MyDrive"
os.environ["TORCH_HOME"] = f"{MY}/torch_cache"   # 사전학습 가중치를 세션마다 받지 않게 둔다
RESULT_DIR = f"{MY}/beyond_mvtec_results"
os.makedirs(RESULT_DIR, exist_ok=True)

MVTEC_CANDIDATES = [f"{MY}/anormaly_detection/mvtec", f"{MY}/mvtec", f"{MY}/MVTecAD"]
MVTEC_DRIVE = next((p for p in MVTEC_CANDIDATES if os.path.isdir(os.path.join(p, "bottle"))), None)
assert MVTEC_DRIVE, f"MVTec 폴더를 못 찾았다. 후보: {MVTEC_CANDIDATES}"

# Drive 에서 파일을 하나씩 읽으면 느려서 런타임 디스크로 옮긴다
MVTEC_LOCAL = "/content/data/mvtec_ad"
if not os.path.isfile(f"{MVTEC_LOCAL}/.copied"):
    os.makedirs(os.path.dirname(MVTEC_LOCAL), exist_ok=True)
    shutil.copytree(MVTEC_DRIVE, MVTEC_LOCAL, dirs_exist_ok=True)
    open(f"{MVTEC_LOCAL}/.copied", "w").close()
print("Drive:", MVTEC_DRIVE, "-> 로컬:", MVTEC_LOCAL)

Mounted at /content/drive
Drive: /content/drive/MyDrive/mvtec -> 로컬: /content/data/mvtec_ad


In [9]:
import pandas as pd
from beyond_mvtec import data, experiment as ex

rows = []
for category in data.MVTEC_CATEGORIES:
    split = data.list_mvtec(MVTEC_LOCAL, category)
    rows.append({"category": category, "train": len(split.train), "test": len(split.test),
                 "test_anomaly": sum(s.label for s in split.test)})
table = pd.DataFrame(rows)
display(table)
# 논문의 장수와 맞는지 확인한다
assert table["train"].sum() == 3629 and table["test"].sum() == 1725, (table["train"].sum(), table["test"].sum())
print("학습 3,629장, 시험 1,725장: 논문과 같다")

,category,train,test,test_anomaly
0,bottle,209,83,63
1,cable,224,150,92
2,capsule,219,132,109
3,carpet,280,117,89
4,grid,264,78,57
5,hazelnut,391,110,70
6,leather,245,124,92
7,metal_nut,220,115,93
8,pill,267,167,141
9,screw,320,160,119


학습 3,629장, 시험 1,725장: 논문과 같다


## 3. 짧은 실행

전체를 돌리기 전에 부류 하나로 GPU 메모리와 faiss GPU가 도는지 본다. 이 값은 재현 판정에 쓰지 않는다.

In [10]:
config = ex.load_config(f"{REPO_DIR}/configs/experiments.yaml")
PREPROCESS = config["preprocess"]["reproduce"]
SETTINGS = ex.reproduce_settings(config)
OPTIONS = dict(device="cuda", batch_size=8, num_workers=4, faiss_on_gpu=True, commit=COMMIT)

smoke = {s["setting"]: s for s in SETTINGS}
for name in ["padim_wrn50_L123_d550", "patchcore_wrn50_L23_c1"]:
    row = ex.run_one(smoke[name], "mvtec_ad", MVTEC_LOCAL, "bottle", 0, "reproduce", PREPROCESS, **OPTIONS)
    print({k: row[k] for k in ["setting", "image_auroc", "pixel_auroc", "pro", "bank_size", "fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]})

{'setting': 'padim_wrn50_L123_d550', 'image_auroc': 0.9992063492063492, 'pixel_auroc': 0.9814888790455276, 'pro': 0.9487178135821976, 'bank_size': '', 'fit_sec': 17.0, 'predict_sec': 1.07, 'metric_sec': 2.03, 'peak_gpu_gb': 14.2}


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 1638/1638 [00:01<00:00, 1104.64it/s]


{'setting': 'patchcore_wrn50_L23_c1', 'image_auroc': 1.0, 'pixel_auroc': 0.9855854018417115, 'pro': 0.9459264436048, 'bank_size': 1638, 'fit_sec': 9.42, 'predict_sec': 2.92, 'metric_sec': 2.1, 'peak_gpu_gb': 1.07}


## 4. 재현 실행

시드 0의 이상 지도는 `maps/` 에 남긴다. PRO가 어긋날 때 정의를 바꿔 다시 재기 위해서다. 부류 하나마다 걸린 시간과 GPU 메모리가 CSV에 남는다.

In [11]:
CSV_PATH = f"{RESULT_DIR}/reproduce_mvtec.csv"
ex.run_grid(SETTINGS, "mvtec_ad", MVTEC_LOCAL, data.MVTEC_CATEGORIES, config["seeds"], "reproduce", PREPROCESS,
            CSV_PATH, save_dir=f"{RESULT_DIR}/maps", save_maps_seeds=(0,), **OPTIONS)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 16385/16385 [00:14<00:00, 1119.41it/s]


[1/180] patchcore_wrn50_L23_c10 bottle s0 I=1.0000 P=0.9860 PRO=0.9468 fit=22.89s pred=3.56s gpu=1.07GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 17561/17561 [00:16<00:00, 1045.81it/s]


[2/180] patchcore_wrn50_L23_c10 cable s0 I=0.9966 P=0.9842 PRO=0.9328 fit=25.92s pred=5.84s gpu=1.15GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 17169/17169 [00:16<00:00, 1059.14it/s]


[3/180] patchcore_wrn50_L23_c10 capsule s0 I=0.9765 P=0.9885 PRO=0.9322 fit=25.3s pred=5.3s gpu=1.11GB


... (172 outputs left out)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 1881/1881 [00:01<00:00, 967.16it/s]


[90/180] patchcore_wrn50_L23_c1 zipper s2 I=0.9971 P=0.9879 PRO=0.9545 fit=11.69s pred=5.29s gpu=1.21GB
[91/180] padim_wrn50_L123_d550 bottle s0 I=0.9992 P=0.9815 PRO=0.9487 fit=11.4s pred=1.2s gpu=14.2GB
[92/180] padim_wrn50_L123_d550 cable s0 I=0.9099 P=0.9663 PRO=0.8867 fit=11.65s pred=2.58s gpu=14.3GB
[93/180] padim_wrn50_L123_d550 capsule s0 I=0.9015 P=0.9848 PRO=0.9149 fit=11.75s pred=2.24s gpu=14.23GB
[94/180] padim_wrn50_L123_d550 carpet s0 I=0.9988 P=0.9893 PRO=0.9606 fit=13.04s pred=2.12s gpu=14.3GB
[95/180] padim_wrn50_L123_d550 grid s0 I=0.9574 P=0.9677 PRO=0.9214 fit=12.5s pred=1.01s gpu=14.3GB
[96/180] padim_wrn50_L123_d550 hazelnut s0 I=0.9132 P=0.9805 PRO=0.9349 fit=15.95s pred=2.0s gpu=14.28GB
[97/180] padim_wrn50_L123_d550 leather s0 I=1.0000 P=0.9895 PRO=0.9710 fit=12.23s pred=2.03s gpu=14.26GB
[98/180] padim_wrn50_L123_d550 metal_nut s0 I=0.9726 P=0.9668 PRO=0.9148 fit=11.4s pred=1.25s gpu=14.24GB
[99/180] padim_wrn50_L123_d550 pill s0 I=0.9348 P=0.9533 PRO=0.9407 f

100%|██████████| 44.7M/44.7M [00:00<00:00, 151MB/s]


[136/180] padim_r18_L123_d100 bottle s0 I=0.9960 P=0.9808 PRO=0.9427 fit=2.86s pred=1.12s gpu=0.52GB
[137/180] padim_r18_L123_d100 cable s0 I=0.8937 P=0.9506 PRO=0.8704 fit=3.3s pred=2.51s gpu=0.54GB
[138/180] padim_r18_L123_d100 capsule s0 I=0.8712 P=0.9834 PRO=0.9114 fit=3.23s pred=2.22s gpu=0.53GB
[139/180] padim_r18_L123_d100 carpet s0 I=0.9848 P=0.9865 PRO=0.9546 fit=3.96s pred=2.04s gpu=0.54GB
[140/180] padim_r18_L123_d100 grid s0 I=0.9407 P=0.9262 PRO=0.8592 fit=2.05s pred=0.93s gpu=0.54GB
[141/180] padim_r18_L123_d100 hazelnut s0 I=0.8104 P=0.9733 PRO=0.9188 fit=5.48s pred=1.96s gpu=0.54GB
[142/180] padim_r18_L123_d100 leather s0 I=0.9990 P=0.9914 PRO=0.9773 fit=3.1s pred=1.93s gpu=0.53GB
[143/180] padim_r18_L123_d100 metal_nut s0 I=0.9780 P=0.9692 PRO=0.9065 fit=1.74s pred=1.15s gpu=0.53GB
[144/180] padim_r18_L123_d100 pill s0 I=0.8860 P=0.9456 PRO=0.9291 fit=2.68s pred=1.88s gpu=0.53GB
[145/180] padim_r18_L123_d100 screw s0 I=0.7821 P=0.9733 PRO=0.9088 fit=2.31s pred=1.42s gp

## 5. 판정

In [12]:
summary = ex.summarize(CSV_PATH)
display(summary.round(3))

tolerance = config["reproduce"]["tolerance"]
verdict = ex.judge_reproduction(summary, SETTINGS, tolerance)
display(verdict)

complete = (verdict["n_seeds"] == 3).all() and (verdict["n_categories"] == 15).all()
judged = verdict[verdict["passed"].notna()]
print("칸이 다 찼나:", complete)
print(f"판정 폭 안: {int((judged['passed'] == True).sum())} / {len(judged)}")
print("다음 단계로 가도 되나:", bool(complete and (judged["passed"] == True).all()))

,dataset,setting,preprocess,n_train_requested,image_auroc_mean,pixel_auroc_mean,pro_mean,image_auroc_std,pixel_auroc_std,pro_std,n_seeds,n_categories
0,mvtec_ad,padim_r18_L123_d100,reproduce,all,92.327,96.336,90.255,0.373,0.343,0.401,3,15
1,mvtec_ad,padim_wrn50_L123_d550,reproduce,all,95.316,97.375,92.389,0.633,0.085,0.190,3,15
2,mvtec_ad,patchcore_wrn50_L23_c1,reproduce,all,98.870,97.950,92.308,0.098,0.023,0.074,3,15
3,mvtec_ad,patchcore_wrn50_L23_c10,reproduce,all,99.000,98.111,92.969,0.056,0.008,0.034,3,15


,setting,metric,target,value,seed_std,diff,tolerance,passed,n_seeds,n_categories
0,patchcore_wrn50_L23_c10,image_auroc,99.0,99.00,0.056,-0.00,0.5,True,3,15
1,patchcore_wrn50_L23_c10,pixel_auroc,98.1,98.11,0.008,0.01,1.0,True,3,15
2,patchcore_wrn50_L23_c10,pro,93.5,92.97,0.034,-0.53,1.0,True,3,15
3,patchcore_wrn50_L23_c1,image_auroc,99.0,98.87,0.098,-0.13,0.5,True,3,15
4,patchcore_wrn50_L23_c1,pixel_auroc,98.0,97.95,0.023,-0.05,1.0,True,3,15
5,patchcore_wrn50_L23_c1,pro,93.1,92.31,0.074,-0.79,1.0,True,3,15
6,padim_wrn50_L123_d550,image_auroc,95.3,95.32,0.633,0.02,0.5,True,3,15
7,padim_wrn50_L123_d550,pixel_auroc,97.5,97.38,0.085,-0.12,1.0,True,3,15
8,padim_wrn50_L123_d550,pro,92.1,92.39,0.190,0.29,1.0,True,3,15
9,padim_r18_L123_d100,image_auroc,NaN,92.33,0.373,NaN,0.5,None,3,15


칸이 다 찼나: True
판정 폭 안: 11 / 11
다음 단계로 가도 되나: True


In [13]:
# 부류별 값(시드 평균, %). PaDiM 은 논문 표 III 의 부류 값과, PatchCore 는 표 S1 과 맞대어 본다.
# PatchCore 의 부류 값은 원본 코드와 논문이 이미 다르므로(Screw 1.77점) 15 부류 평균 폭을 부류별로 쓰지 않는다.
df = pd.read_csv(CSV_PATH)
for metric in ex.METRICS:
    print(metric)
    display((df.pivot_table(index="category", columns="setting", values=metric, aggfunc="mean") * 100).round(2))
print("시간(초), 설정별 부류 평균")
display(df.groupby("setting")[["fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]].mean().round(1))

image_auroc


setting,padim_r18_L123_d100,padim_wrn50_L123_d550,patchcore_wrn50_L23_c1,patchcore_wrn50_L23_c10
category,,,,
bottle,99.71,99.89,100.00,100.00
cable,87.82,91.20,99.39,99.66
capsule,85.72,90.81,96.72,97.33
carpet,99.09,99.91,98.46,98.62
grid,92.56,96.27,98.44,97.74
hazelnut,83.68,92.57,100.00,100.00
leather,99.97,100.00,100.00,100.00
metal_nut,98.34,98.48,99.87,99.93
pill,86.33,92.74,97.12,96.44


pixel_auroc


setting,padim_r18_L123_d100,padim_wrn50_L123_d550,patchcore_wrn50_L23_c1,patchcore_wrn50_L23_c10
category,,,,
bottle,97.95,98.20,98.55,98.60
cable,95.08,96.61,98.18,98.44
capsule,98.14,98.51,98.82,98.85
carpet,98.73,98.95,98.96,98.99
grid,93.47,97.16,98.43,98.66
hazelnut,97.30,98.11,98.60,98.68
leather,99.08,99.06,99.28,99.28
metal_nut,96.19,96.68,98.30,98.38
pill,94.02,95.76,97.45,97.61


pro


setting,padim_r18_L123_d100,padim_wrn50_L123_d550,patchcore_wrn50_L23_c1,patchcore_wrn50_L23_c10
category,,,,
bottle,94.09,94.97,94.65,94.75
cable,87.12,88.69,91.62,93.09
capsule,90.43,91.64,92.33,93.50
carpet,95.80,96.24,94.67,94.96
grid,86.55,92.61,93.18,93.59
hazelnut,91.59,93.50,94.84,94.64
leather,97.70,97.52,97.03,97.15
metal_nut,89.88,91.69,93.08,93.96
pill,92.52,94.52,94.55,94.98


시간(초), 설정별 부류 평균


,fit_sec,predict_sec,metric_sec,peak_gpu_gb
setting,,,,
padim_r18_L123_d100,2.9,1.6,3.1,0.5
padim_wrn50_L123_d550,12.1,1.7,3.1,14.3
patchcore_wrn50_L23_c1,12.2,4.2,3.1,1.2
patchcore_wrn50_L23_c10,31.3,4.6,3.0,1.2


## 6. 결과를 레포로

`reproduce_mvtec.csv`는 Drive의 `beyond_mvtec_results/`에 있다. 이 파일을 레포의 `results/`에 넣고 커밋한다. 지도(`maps/`)는 레포에 올리지 않는다.